# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

> Croissant Schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata (Croissant schema)
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, their `@id`'s, and columns. Reference all entities by their `@id` as per FAIR best practices.

**Available record sets:**

In [ ]:
# List all record sets by @id and show field & column @id's within each
record_sets = list(dataset.record_sets)
print(f"Number of available RecordSets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name')}")
    fields = rs.get('fields', [])
    if not fields:
        print("    (No field definitions loaded in metadata. They may be inlined in data file.)")
    else:
        print("  Fields: (by @id)")
        for field in fields:
            print(f"    - {field['@id']}")
            cols = field.get('columns', [])
            if cols:
                print(f"        Columns in field:")
                for col in cols:
                    print(f"          - {col['@id']}")
    print("")
# Preview a few records from the first record set
if len(record_sets) > 0:
    first_record_set_id = record_sets[0]['@id']
    print(f"\nPreview first 3 records for record set: {first_record_set_id}\n")
    for i, x in enumerate(dataset.records(record_set=first_record_set_id)):
        print(x)
        if i >= 2:
            break

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Prepare to extract data for each available record set
import pprint
# Gather record set ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Listify iterator
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded RecordSet {record_set_id}: {df.shape[0]} rows\n")
            print(f"Columns: {df.columns.tolist()}\n")
            print(df.head(), "\n")
        else:
            print(f"RecordSet {record_set_id} yielded no records.")
    except Exception as e:
        print(f"Could not retrieve records for {record_set_id}: {type(e).__name__}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps to a sample numeric field from the main data record set. Operations include filtering, normalization, and grouping.

Choose fields and columns by their `@id` as shown above.

In [ ]:
# For demonstration, pick main record set and a numeric column by @id
# (You may replace these @ids if your exploration above found different ones.)

main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
df = dataframes.get(main_record_set_id)

# Try to auto-detect likely numeric column
numeric_field = None
if df is not None:
    for col in df.columns:
        # Try first column with int/float values
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
if not numeric_field and df is not None:
    # Try conversion
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            pass

if df is not None and numeric_field:
    print(f"Using numeric column '{numeric_field}' for EDA.")
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    # Filtering: show records with numeric_field above threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} ({len(filtered_df)} records):")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a likely categorical column
    group_field = None
    for col in df.columns:
        if col != numeric_field and pd.api.types.is_string_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("Could not perform EDA: No suitable numeric field was found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib to plot the distribution of the chosen numeric field, and a group-wise mean barplot if grouped data is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

    if 'grouped_df' in locals() and group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df.index, y=grouped_df.values)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("Visualization skipped: No suitable numeric field loaded.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset on mass spectrometry of extracellular vesicles from human mast cells using the `mlcroissant` library. We inspected available record sets and fields by their `@id`, loaded tables into pandas DataFrames, performed basic exploratory data analysis (EDA), and visualized key data distributions.

This approach can be extended to more complex analyses, including advanced filtering, feature engineering, and machine learning tasks, by leveraging the structured metadata and well-described fields provided by the Croissant schema.

For further exploration, refer to the dataset's full Croissant schema and documentation to understand the meaning and provenance of each field and utilize them for reproducible, FAIR data science.